# 📊 01 – Statistiques avec SciPy

---

## 🎯 Objectifs
- Comprendre ce qu'est SciPy et pourquoi on l'utilise en plus de NumPy
- Générer des données aléatoires réalistes avec `scipy.stats`
- Calculer et interpréter les statistiques descriptives
- Comprendre et réaliser un **test statistique** (t-test) sans formule mathématique
- Interpréter une **p-value** en langage humain

> ℹ️ **Astuce** : Les corrections sont cachées dans des blocs pliables.  
> Cliquez sur **💡 Correction** pour dérouler la solution.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
print("SciPy prêt ✅")

---
## 📝 Partie 0 – Qu'est-ce que SciPy ?

### 🔎 NumPy vs SciPy — quelle différence ?

On a déjà NumPy pour les calculs numériques. Pourquoi SciPy en plus ?

```
NumPy  → les briques de base : tableaux, opérations mathématiques, statistiques simples
SciPy  → les outils avancés construits au-dessus de NumPy :
          - statistiques avancées (tests, distributions)
          - optimisation (trouver le meilleur paramètre)
          - intégration (calculer des aires)
          - traitement du signal (analyser des sons, capteurs)
```

**Analogie :** NumPy c'est la boîte à outils de base (marteau, tournevis). SciPy c'est la perceuse électrique — même famille, mais pour des travaux plus spécialisés.

### 🔎 Les modules SciPy qu'on va voir

| Module | Ce qu'il fait | Ce notebook |
|--------|--------------|-------------|
| `scipy.stats` | Statistiques, tests, distributions | ✅ Ce notebook |
| `scipy.optimize` | Trouver le meilleur paramètre | → Notebook 02 |
| `scipy.integrate` | Calculer des aires, simuler | → Notebook 03 |
| `scipy.fft` + `scipy.signal` | Analyser des signaux | → Notebook 04 |

---
## 📝 Partie 1 – Générer des données réalistes

### 🔎 `np.random.normal()` — la distribution normale

Dans la vraie vie, beaucoup de phénomènes suivent une **distribution normale** (courbe en cloche) : les tailles humaines, les notes d'un examen, les temps de traitement d'une machine.

```
np.random.normal(loc, scale, size)
                  ↑      ↑      ↑
               moyenne  écart  nombre
                        type   de valeurs

Exemple :
notes = np.random.normal(loc=12, scale=2, size=30)
→ génère 30 notes aléatoires autour de 12, dispersées de ±2 points
→ la plupart seront entre 10 et 14, quelques-unes entre 8 et 16
```

> 💡 **`np.random.seed(n)`** : fixe le générateur aléatoire pour obtenir **toujours les mêmes résultats**. Indispensable pour que tout le monde ait les mêmes données en formation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

# Classe A : bons résultats, moyenne 14, peu dispersés
notes_A = np.random.normal(loc=14, scale=1.5, size=30)
# Classe B : résultats moyens, moyenne 11, plus dispersés
notes_B = np.random.normal(loc=11, scale=2.5, size=30)

print("=== Classe A ===")
print(f"  Moyenne  : {notes_A.mean():.2f}")
print(f"  Std      : {notes_A.std():.2f}")
print(f"  Min / Max: {notes_A.min():.1f} / {notes_A.max():.1f}")

print("\n=== Classe B ===")
print(f"  Moyenne  : {notes_B.mean():.2f}")
print(f"  Std      : {notes_B.std():.2f}")
print(f"  Min / Max: {notes_B.min():.1f} / {notes_B.max():.1f}")

# Visualiser les distributions
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(notes_A, bins=10, alpha=0.7, color="steelblue", label="Classe A (moy=14)")
ax.hist(notes_B, bins=10, alpha=0.7, color="coral",     label="Classe B (moy=11)")
ax.axvline(notes_A.mean(), color="steelblue", linestyle="--", linewidth=2)
ax.axvline(notes_B.mean(), color="coral",     linestyle="--", linewidth=2)
ax.set_title("Distribution des notes — deux classes")
ax.set_xlabel("Note")
ax.set_ylabel("Nombre d'élèves")
ax.legend()
plt.tight_layout()
plt.show()

---
## 📝 Partie 2 – Statistiques descriptives avec SciPy

### 🔎 `stats.describe()` — le résumé complet

SciPy offre une fonction `describe()` qui donne en une seule ligne toutes les statistiques descriptives importantes :

```python
result = stats.describe(donnees)
result.nobs      # nombre de valeurs
result.mean      # moyenne
result.variance  # variance
result.skewness  # asymétrie (0 = symétrique, >0 = queue à droite)
result.kurtosis  # "pic" de la distribution (0 = normale)
result.minmax    # (minimum, maximum)
```

### 🔎 Skewness et Kurtosis — deux notions utiles

**Skewness (asymétrie) :** indique si la distribution est penchée d'un côté
```
Skewness = 0   → distribution symétrique (courbe en cloche parfaite)
Skewness > 0   → queue à droite (quelques valeurs très hautes)
               ex: salaires (quelques PDG tirent la moyenne vers le haut)
Skewness < 0   → queue à gauche (quelques valeurs très basses)
```

**Kurtosis :** indique si la distribution est très "pointue" ou très "plate"
```
Kurtosis ≈ 0   → distribution normale standard
Kurtosis > 0   → distribution très pointue (valeurs très concentrées)
Kurtosis < 0   → distribution plate (valeurs très étalées)
```

In [ ]:
import numpy as np
from scipy import stats
np.random.seed(42)

notes_A = np.random.normal(loc=14, scale=1.5, size=30)
notes_B = np.random.normal(loc=11, scale=2.5, size=30)

# stats.describe() — résumé complet
desc_A = stats.describe(notes_A)
desc_B = stats.describe(notes_B)

print("=== Résumé Classe A ===")
print(f"  Nb valeurs : {desc_A.nobs}")
print(f"  Min / Max  : {desc_A.minmax[0]:.2f} / {desc_A.minmax[1]:.2f}")
print(f"  Moyenne    : {desc_A.mean:.2f}")
print(f"  Variance   : {desc_A.variance:.2f}")
print(f"  Skewness   : {desc_A.skewness:.3f}  (proche de 0 → symétrique)")
print(f"  Kurtosis   : {desc_A.kurtosis:.3f}")

print("\n=== Résumé Classe B ===")
print(f"  Nb valeurs : {desc_B.nobs}")
print(f"  Min / Max  : {desc_B.minmax[0]:.2f} / {desc_B.minmax[1]:.2f}")
print(f"  Moyenne    : {desc_B.mean:.2f}")
print(f"  Variance   : {desc_B.variance:.2f}")
print(f"  Skewness   : {desc_B.skewness:.3f}")
print(f"  Kurtosis   : {desc_B.kurtosis:.3f}")

---
## 📝 Partie 3 – Le test statistique : est-ce que la différence est réelle ?

### 🔎 Le problème concret

La Classe A a une moyenne de 14, la Classe B une moyenne de 11. La différence est **visible**. Mais est-elle **réelle** ou juste due au hasard ?

**Exemple :** si on tire deux groupes de 5 personnes au hasard dans la même population, leurs moyennes seront quand même un peu différentes par chance. Plus l'échantillon est petit, plus les différences "au hasard" peuvent être grandes.

Un **test statistique** répond à la question : *"La différence qu'on observe pourrait-elle s'expliquer uniquement par le hasard ?"*

### 🔎 Le t-test — comparer deux groupes

Le **t-test** est le test le plus courant pour comparer les moyennes de deux groupes. Il prend en compte :
- la différence de moyennes
- la dispersion des données
- la taille des échantillons

```python
t_stat, p_value = stats.ttest_ind(groupe1, groupe2)
#                                   ↑ ind = independent = les deux groupes sont séparés
```

### 🔎 La p-value — comment l'interpréter ?

La **p-value** est la probabilité que la différence observée soit due **uniquement au hasard** si les deux groupes étaient en réalité identiques.

```
p-value = 0.80  → 80% de chances que ce soit du hasard → différence NON significative
p-value = 0.30  → 30% de chances que ce soit du hasard → différence NON significative
p-value = 0.04  → 4% de chances que ce soit du hasard  → différence SIGNIFICATIVE ✅
p-value = 0.001 → 0.1% de chances                       → différence TRÈS significative ✅

Règle universelle : si p-value < 0.05 → la différence est considérée réelle
                    si p-value ≥ 0.05 → on ne peut pas conclure
```

> 💡 **En langage humain :** une p-value de 0.04 signifie "Si les deux classes étaient en réalité au même niveau, il n'y aurait que 4% de chance qu'on observe une différence aussi grande par pur hasard. Donc on conclut que la différence est réelle."

In [ ]:
import numpy as np
from scipy import stats
np.random.seed(42)

notes_A = np.random.normal(loc=14, scale=1.5, size=30)
notes_B = np.random.normal(loc=11, scale=2.5, size=30)

# t-test : est-ce que la différence de moyennes est significative ?
t_stat, p_value = stats.ttest_ind(notes_A, notes_B)

print(f"Moyenne Classe A : {notes_A.mean():.2f}")
print(f"Moyenne Classe B : {notes_B.mean():.2f}")
print(f"Différence       : {notes_A.mean() - notes_B.mean():.2f} points")
print()
print(f"t-statistique    : {t_stat:.3f}")
print(f"p-value          : {p_value:.6f}")
print()

# Interprétation automatique
seuil = 0.05
if p_value < seuil:
    print(f"✅ p-value ({p_value:.4f}) < {seuil} → différence SIGNIFICATIVE")
    print("   La Classe A est statistiquement meilleure que la Classe B.")
else:
    print(f"❌ p-value ({p_value:.4f}) ≥ {seuil} → différence NON significative")
    print("   On ne peut pas conclure que les deux classes sont différentes.")

---
## 📝 Partie 4 – Autres tests statistiques utiles

### 🔎 Quand utiliser quel test ?

| Situation | Test | Fonction SciPy |
|-----------|------|----------------|
| Comparer **2 groupes indépendants** | t-test | `stats.ttest_ind(A, B)` |
| Comparer **avant/après** sur le même groupe | t-test apparié | `stats.ttest_rel(avant, apres)` |
| Comparer **3 groupes ou plus** | ANOVA | `stats.f_oneway(A, B, C)` |
| Vérifier si des données suivent une **loi normale** | Test de Shapiro | `stats.shapiro(donnees)` |
| Mesurer la **corrélation** entre deux variables | Pearson | `stats.pearsonr(x, y)` |

### 🔎 La corrélation de Pearson

On l'a vue avec NumPy (`np.corrcoef`). SciPy va plus loin : il donne aussi la **p-value** pour savoir si la corrélation est significative.

```python
r, p = stats.pearsonr(x, y)
# r = coefficient de corrélation (-1 à +1)
# p = p-value (la corrélation est-elle réelle ou due au hasard ?)
```

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
np.random.seed(7)

# === Test de normalité (Shapiro-Wilk) ===
# Question : "Mes données suivent-elles une loi normale ?"
notes = np.random.normal(12, 2, 30)
stat_shapiro, p_shapiro = stats.shapiro(notes)

print("=== Test de normalité (Shapiro-Wilk) ===")
print(f"p-value : {p_shapiro:.4f}")
if p_shapiro > 0.05:
    print("✅ Les données suivent une loi normale (p > 0.05)")
else:
    print("❌ Les données NE suivent PAS une loi normale (p < 0.05)")

print()

# === Corrélation de Pearson ===
# Question : "Les heures de travail sont-elles liées aux notes ?"
heures_travail = np.array([2, 3, 5, 4, 6, 7, 3, 8, 5, 6, 4, 9, 7, 5, 8])
notes_examen   = np.array([8, 10, 13, 11, 15, 16, 9, 18, 12, 14, 11, 19, 17, 12, 17])

r, p_corr = stats.pearsonr(heures_travail, notes_examen)

print("=== Corrélation : heures de travail ↔ note ===")
print(f"Coefficient r : {r:.3f}")
print(f"p-value       : {p_corr:.6f}")
if p_corr < 0.05:
    print(f"✅ Corrélation significative : plus on travaille, meilleures sont les notes (r={r:.2f})")

# Visualisation
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(heures_travail, notes_examen, color="steelblue", s=80, alpha=0.8)
ax.set_title(f"Heures de travail vs Note — r = {r:.2f}, p = {p_corr:.4f}")
ax.set_xlabel("Heures de travail")
ax.set_ylabel("Note à l'examen")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from scipy import stats
np.random.seed(3)

# === ANOVA : comparer 3 groupes ou plus ===
# Situation : 3 méthodes de formation — laquelle est la plus efficace ?
methode_A = np.random.normal(72, 8, 25)   # formation classique
methode_B = np.random.normal(78, 7, 25)   # formation en ligne
methode_C = np.random.normal(85, 6, 25)   # formation mixte

f_stat, p_anova = stats.f_oneway(methode_A, methode_B, methode_C)

print("=== ANOVA : comparaison de 3 méthodes de formation ===")
print(f"Score moyen méthode A (classique) : {methode_A.mean():.1f}")
print(f"Score moyen méthode B (en ligne)  : {methode_B.mean():.1f}")
print(f"Score moyen méthode C (mixte)     : {methode_C.mean():.1f}")
print()
print(f"F-statistique : {f_stat:.3f}")
print(f"p-value       : {p_anova:.6f}")

if p_anova < 0.05:
    print("\n✅ Différence significative entre les méthodes (p < 0.05)")
    print("   Au moins une méthode est statistiquement meilleure que les autres.")
    print("   → La méthode mixte (C) semble la plus efficace.")
else:
    print("\n❌ Pas de différence significative entre les méthodes.")

---
## 🧩 Exercice 1 – Comparer deux équipes commerciales

Votre direction veut savoir si l'équipe A performe mieux que l'équipe B.

```python
np.random.seed(10)
equipe_A = np.random.normal(loc=85, scale=10, size=25)  # CA moyen en k€
equipe_B = np.random.normal(loc=78, scale=12, size=25)
```

1. Calculez la moyenne et l'écart-type de chaque équipe
2. Visualisez les deux distributions en histogrammes superposés
3. Réalisez un t-test et interprétez la p-value
4. La différence est-elle statistiquement significative ?
5. Testez si les données de l'équipe A suivent une loi normale (Shapiro-Wilk)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(10)
equipe_A = np.random.normal(loc=85, scale=10, size=25)
equipe_B = np.random.normal(loc=78, scale=12, size=25)

# TODO 1 : statistiques descriptives
print(f"Équipe A — Moyenne : {equipe_A.mean():.1f} k€ | Std : {equipe_A.std():.1f}")
print(f"Équipe B — Moyenne : {equipe_B.mean():.1f} k€ | Std : {equipe_B.std():.1f}")

# TODO 2 : histogrammes superposés
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(equipe_A, bins=..., alpha=0.7, color=..., label="Équipe A")
ax.hist(equipe_B, bins=..., alpha=0.7, color=..., label="Équipe B")
ax.axvline(equipe_A.mean(), color=..., linestyle="--")
ax.axvline(equipe_B.mean(), color=..., linestyle="--")
ax.set_title("Distribution des performances")
ax.set_xlabel("CA (k€)")
ax.legend()
plt.tight_layout()
plt.show()

# TODO 3 : t-test
t_stat, p_value = stats.ttest_ind(..., ...)
print(f"\nt-statistique : {t_stat:.3f}")
print(f"p-value       : {p_value:.4f}")

# TODO 4 : interprétation
if p_value < 0.05:
    print("✅ Différence significative")
else:
    print("❌ Différence non significative")

# TODO 5 : test de normalité
_, p_shapiro = stats.shapiro(...)
print(f"\nShapiro (Équipe A) : p = {p_shapiro:.4f}")
print("→ Distribution normale ?" , "Oui" if p_shapiro > 0.05 else "Non")

<details>
<summary>💡 Correction</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(10)
equipe_A = np.random.normal(loc=85, scale=10, size=25)
equipe_B = np.random.normal(loc=78, scale=12, size=25)

# 1
print(f"Équipe A — Moyenne : {equipe_A.mean():.1f} k€ | Std : {equipe_A.std():.1f}")
print(f"Équipe B — Moyenne : {equipe_B.mean():.1f} k€ | Std : {equipe_B.std():.1f}")

# 2
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(equipe_A, bins=8, alpha=0.7, color="steelblue", label="Équipe A")
ax.hist(equipe_B, bins=8, alpha=0.7, color="coral",     label="Équipe B")
ax.axvline(equipe_A.mean(), color="steelblue", linestyle="--")
ax.axvline(equipe_B.mean(), color="coral",     linestyle="--")
ax.set_title("Distribution des performances")
ax.set_xlabel("CA (k€)")
ax.legend()
plt.tight_layout()
plt.show()

# 3
t_stat, p_value = stats.ttest_ind(equipe_A, equipe_B)
print(f"t-statistique : {t_stat:.3f}")
print(f"p-value       : {p_value:.4f}")

# 4 — p-value > 0.05 probablement → différence visible mais pas significative
# avec seulement 25 personnes par groupe et un écart de 7 k€, le hasard peut l'expliquer
if p_value < 0.05:
    print("✅ Différence significative — Équipe A meilleure")
else:
    print("❌ Pas de différence significative — l'écart peut être dû au hasard")

# 5
_, p_shapiro = stats.shapiro(equipe_A)
print(f"Shapiro (Équipe A) : p = {p_shapiro:.4f}")
print("Distribution normale :", "Oui" if p_shapiro > 0.05 else "Non")
```
</details>

---
## 🧩 Exercice 2 – Avant / Après une formation

Une entreprise a formé 15 commerciaux. On veut savoir si la formation a amélioré leurs résultats.

```python
np.random.seed(5)
avant  = np.random.normal(65, 8, 15)           # scores avant formation
apres  = avant + np.random.normal(8, 5, 15)    # scores après (amélioration moyenne de +8)
```

1. Calculez la moyenne avant et après
2. Calculez l'amélioration moyenne par personne
3. Utilisez un **t-test apparié** (`ttest_rel`) — car c'est le **même groupe** avant/après
4. La formation a-t-elle eu un effet significatif ?
5. Visualisez avant/après avec deux histogrammes

> 💡 **Pourquoi `ttest_rel` et pas `ttest_ind` ?**  
> `ttest_ind` compare deux groupes **différents** (Classe A vs Classe B).  
> `ttest_rel` compare le **même groupe** à deux moments différents (avant vs après). C'est plus puissant car il élimine les différences individuelles.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(5)
avant = np.random.normal(65, 8, 15)
apres = avant + np.random.normal(8, 5, 15)

# TODO 1 : moyennes
print(f"Score moyen AVANT : {avant.mean():.1f}")
print(f"Score moyen APRÈS : {apres.mean():.1f}")

# TODO 2 : amélioration par personne
amelioration = apres - avant
print(f"Amélioration moyenne : +{amelioration.mean():.1f} points")

# TODO 3 : t-test apparié
t_stat, p_value = stats.ttest_rel(..., ...)
print(f"\nt-stat : {t_stat:.3f} | p-value : {p_value:.6f}")

# TODO 4 : interprétation
if p_value < 0.05:
    print("✅ La formation a eu un effet significatif")
else:
    print("❌ L'effet n'est pas significatif")

# TODO 5 : visualisation
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(avant, bins=7, alpha=0.7, color="coral",     label=f"Avant (moy={avant.mean():.1f})")
ax.hist(apres, bins=7, alpha=0.7, color="seagreen",  label=f"Après (moy={apres.mean():.1f})")
ax.set_title("Scores avant et après la formation")
ax.set_xlabel("Score")
ax.legend()
plt.tight_layout()
plt.show()

<details>
<summary>💡 Correction</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(5)
avant = np.random.normal(65, 8, 15)
apres = avant + np.random.normal(8, 5, 15)

print(f"Score moyen AVANT : {avant.mean():.1f}")
print(f"Score moyen APRÈS : {apres.mean():.1f}")

amelioration = apres - avant
print(f"Amélioration moyenne : +{amelioration.mean():.1f} points")

# ttest_rel car même groupe avant/après
t_stat, p_value = stats.ttest_rel(avant, apres)
print(f"t-stat : {t_stat:.3f} | p-value : {p_value:.6f}")

# p-value très faible → amélioration réelle, pas due au hasard
if p_value < 0.05:
    print("✅ La formation a eu un effet significatif")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(avant, bins=7, alpha=0.7, color="coral",    label=f"Avant (moy={avant.mean():.1f})")
ax.hist(apres, bins=7, alpha=0.7, color="seagreen", label=f"Après (moy={apres.mean():.1f})")
ax.set_title("Scores avant et après la formation")
ax.set_xlabel("Score")
ax.legend()
plt.tight_layout()
plt.show()
```
</details>

---
## ✅ Récapitulatif

| Concept | Ce qu'il faut retenir |
|---------|----------------------|
| **`scipy.stats`** | Module SciPy pour les statistiques avancées |
| **`np.random.normal(loc, scale, size)`** | Génère des données aléatoires réalistes autour d'une moyenne |
| **`stats.describe(x)`** | Résumé complet : moyenne, variance, skewness, kurtosis |
| **`stats.ttest_ind(A, B)`** | Compare 2 groupes **différents** — est-ce que la différence est réelle ? |
| **`stats.ttest_rel(avant, apres)`** | Compare le **même groupe** à deux moments — avant/après |
| **`stats.f_oneway(A, B, C)`** | Compare **3 groupes ou plus** (ANOVA) |
| **`stats.pearsonr(x, y)`** | Corrélation + p-value — le lien est-il significatif ? |
| **`stats.shapiro(x)`** | Les données suivent-elles une loi normale ? |
| **p-value < 0.05** | La différence est considérée **réelle** (pas due au hasard) |
| **p-value ≥ 0.05** | On **ne peut pas conclure** — l'écart peut être dû au hasard |

**La règle d'or :** la p-value ne dit pas "la différence est grande", elle dit "la différence est réelle". Une petite différence peut être très significative si l'échantillon est grand.

---
📘 **Prochain notebook → `02` : Optimisation avec SciPy**